In [ ]:
from pathlib import Path
import json,sys,base64,html,ast,re,difflib
from IPython.display import display,HTML
P=Path('/home/ec2-user/SageMaker/election_arm');B=P/'source_review_150_v1';B.mkdir(exist_ok=True)
pack=json.loads(base64.b64decode('...'))
p=B/'pack.json'
if p.exists():assert json.loads(p.read_text())==pack
else:p.write_text(json.dumps(pack,indent=2))
sys.path.insert(0,str(P/'source_review_pilot_tdm_v1'))
sys.path.insert(0,str(P/'source_review_pilot_tdm_v1/widget_vendor'))
from proquest_tdm_review import recover,atomic_json
import ipywidgets as w
if (B/'sources.json').exists():sources=json.loads((B/'sources.json').read_text())
else:
 old={s['audit_id']:s for s in json.loads((P/'source_review_main_ai_v1/sources.json').read_text())}
 sources=[]
 for i,r in enumerate(pack['records'],1):
  sources.append(old[r['audit_id']] if r['audit_id'] in old else recover(P,r))
  if i%10==0:print('Retrieved',i,flush=True)
 atomic_json(B/'sources.json',sources)
assert len(sources)==150 and all(s['recovered'] for s in sources)
print('Recovered 150/150; source text remains in VM.')
tree=ast.parse((P/'extract_gpt.py').read_text())
prompts=[x.value for x in ast.walk(tree) if isinstance(x,ast.Constant) and isinstance(x.value,str) and 'FALSIFIABLE' in x.value]
print('Extraction prompt blocks:',len(prompts))
if prompts:
 (B/'extraction_prompt.txt').write_text(prompts[0])
 prompt_idx={'i':0};po=w.HTML();pn=w.Button(description='Next instructions');pp=w.Button(description='Previous instructions')
 def pr():po.value='<h3>Original extraction instructions '+str(prompt_idx['i']+1)+'</h3><pre style="white-space:pre-wrap;font:14px/1.35 Arial">'+html.escape(prompts[0][prompt_idx['i']*2500:(prompt_idx['i']+1)*2500])+'</pre>'
 def pm(d):prompt_idx['i']=max(0,min((len(prompts[0])-1)//2500,prompt_idx['i']+d));pr()
 pn.on_click(lambda _:pm(1));pp.on_click(lambda _:pm(-1));pr();display(w.VBox([w.HBox([pp,pn]),po]))


In [ ]:
state={'i':0,'offset':0,'begin':False}
head=w.HTML();body=w.HTML()
prev=w.Button(description='Previous 2');nxt=w.Button(description='Next 2');earlier=w.Button(description='Earlier context');later=w.Button(description='Later context');begin=w.Button(description='Article beginning');reset=w.Button(description='Quote context')
def clean(t):return re.sub(r'\s+',' ',t).strip()
def render():
 cards=[]
 for i in range(state['i'],min(state['i']+2,150)):
  r,s=pack['records'][i],sources[i];q,t=clean(s['quote']),clean(s['article']);pos=t.find(q)
  if pos<0:
   m=difflib.SequenceMatcher(None,q.lower(),t.lower(),autojunk=False).find_longest_match();pos=max(0,m.b-m.a)
  start=max(0,(0 if state['begin'] else pos-400)+state['offset']);context=t[start:start+1800]
  cards.append('<div style="width:50%;padding:0 10px;box-sizing:border-box;font:13px/1.35 Arial"><h3>'+str(i+1)+'. '+r['date']+' | '+r['audit_id']+'</h3><p>'+html.escape(r['publisher'])+'</p><b>CLAIM</b><p>'+html.escape(q)+'</p><b>CONTEXT '+str(start)+':'+str(start+len(context))+' / '+str(len(t))+'</b><p>'+html.escape(context)+'</p></div>')
 head.value='<b>Claim labeling '+str(state['i']+1)+'–'+str(state['i']+2)+' / 150</b> | 150 sources recovered | Comparison labels hidden'
 body.value='<div style="display:flex">'+''.join(cards)+'</div>'
def move(d):state.update(i=max(0,min(148,state['i']+d)),offset=0,begin=False);render()
def offset(d):state['offset']+=d;render()
def mode(b):state.update(begin=b,offset=0);render()
prev.on_click(lambda _:move(-2));nxt.on_click(lambda _:move(2));earlier.on_click(lambda _:offset(-1400));later.on_click(lambda _:offset(1400));begin.on_click(lambda _:mode(True));reset.on_click(lambda _:mode(False))
render();display(w.VBox([head,w.HBox([prev,nxt,earlier,later,begin,reset]),body]))
